# Governance Dashboard — QuickSight Analysis for Drift Monitoring

> 📊 **Note**: this notebook contains interactive output cells (Evidently reports, plotly charts). They render fully in JupyterLab; static viewers like the GitHub/GitLab renderer strip the JavaScript.


This notebook programmatically creates a complete QuickSight dashboard — no manual UI steps.

**Schema awareness:** the feature-drift dataset (Sheet 3) and the inference dataset both expose `monitoring_run_id`, and the datasets join on it rather than on a time window. Lab 5D stamps the run id onto every inference row it scored, so run membership is recorded at scoring time instead of being inferred from timestamps. The join is therefore exact at any cadence: a run whose window opened weeks ago still matches its own rows, and back-to-back runs minutes apart cannot double-count, because each inference row carries exactly one run's id however much the runs' time ranges overlap.

**Visuals — Sheet 1 (Model Drift Trends):**
1. ROC-AUC: Baseline vs Current Over Time
2. Model Performance Metrics Over Time
3. ROC-AUC Trend by Model Version
4. Model-Drift Verdict Rate by Model Package ARN
5. Model-Drift Verdict Rate by Endpoint
6. Performance by Training Snapshot
7. Model Lineage Audit (per monitoring run)
8. Latest Current ROC-AUC (KPI)
9. Confusion Matrix Over Time (TP/FP/TN/FN)
10. ROC-AUC Degradation % Over Time
11. Ground-Truth Coverage Over Time (labels / predictions)

**Visuals — Sheet 2 (Data Drift Trends):**
1. Data Drift Share Over Time
2. Drifted Features Count Over Time
3. Drift Alerts Timeline
4. Data Drift Share by Model Version
5. Data Drift Share by Endpoint
6. Inference Volume vs Drift Share Correlation
7. Latest Data Drift Share (KPI)
8. Source Data — monitoring_responses (table)
9. Prediction Score Distribution Over Time (leading indicator)
10. Drift Verdict Sample Size Per Run

**Visuals — Sheet 3 (Feature Drift Trends):**
1. Feature Drift Score Timeline (per-feature line chart)
2. Top 15 Most-Drifting Features (All Time)
3. Drift Severity Distribution by Feature (Top 15)
4. Feature Drift Heatmap (Features × Time)
5. Feature Drift Details (per run × feature table)
6. Highest Current Drift Score (KPI)
7. Feature Drift Heatmap (Features × Model Version)
8. Max Feature Drift Score Per Run (worst-feature signal)
9. Repeat-Offender Features (# of runs where drift_detected=TRUE)
10. Raw drift_score — p-value tests (KS / Chi-square) — LOWER = more drift
11. Raw drift_score — distance tests (Wasserstein / Jensen-Shannon / PSI) — HIGHER = more drift

## What this notebook does

This notebook builds (or refreshes) the QuickSight **governance dashboard** — the
inference, drift, feature-drift, and prediction-accuracy visuals — on top of the
Athena tables written by the drift-monitoring pipeline.

**All of the logic lives in one place:** `src/governance/create_governance_dashboard.py`.
That module is the single source of truth for every dataset, Athena view, visual, analysis,
and dashboard definition. It is also what `main.py dashboard create` calls. This notebook is
a thin driver over that module — it does **not** redefine any datasets or visuals inline, so
the notebook and the CLI can never drift out of sync.
`src/governance/dashboard_notebook.py` only adds the notebook-friendly error
reporting and printing around it.

> Because everything is defined in the module, the feature-drift visuals automatically use the
> test-agnostic **`drift_magnitude`** field (higher = more drift, regardless of which statistical
> test Evidently picked) instead of the ambiguous raw `drift_score`.

## 1. Setup & Configuration

In [ ]:
# Load the shared .env (written by lab0-setup/setup.ipynb) and clear stale src
# modules, then import the governance module and its notebook wrappers.
from src.utils.notebook_env import bootstrap
env = bootstrap()

from src.config.config import AWS_DEFAULT_REGION
import src.governance.dashboard_notebook as dash

ACCOUNT_ID = dash.describe_environment(env['project_name'])

> ⚠️ **QuickSight subscription required (once per AWS account).**
>
> The account must be subscribed to QuickSight **Enterprise** before this notebook can create anything — the Definition API it publishes through is an Enterprise feature. The next cell reports the edition it finds.
>
> Setup has two halves, and they are separate events on purpose:
>
> 1. **subscribe the account**, which is what creates the `aws-quicksight-service-role-v0` service role;
> 2. **grant that role access to the data**, by deploying with `AttachQuickSightServiceRolePolicy=true` (`QuickSightServiceRoleS3Policy` / `QuickSightServiceRoleAthenaPolicy` in `templates/2-iam.yaml`).
>
> The order is forced: an IAM policy resource fails outright if the role it targets does not exist, which is why the parameter defaults to `false`. And skipping the second half is the usual cause of a dashboard that publishes successfully but whose every panel fails to load — QuickSight reads Athena and S3 **as its own service role**, not as this notebook's execution role.
>
> Two commands tell you where you stand:
>
> ```bash
> ACCOUNT_ID="$(aws sts get-caller-identity --query Account --output text)"
>
> aws quicksight describe-account-settings --aws-account-id "${ACCOUNT_ID}" \
>   --region us-east-1 --query 'AccountSettings.Edition' --output text
>
> aws iam list-role-policies --role-name aws-quicksight-service-role-v0 \
>   --query 'PolicyNames' --output text
> ```
>
> You want `ENTERPRISE` from the first and both `<ProjectName>-QuickSightS3DataLakeAccess` and `<ProjectName>-QuickSightAthenaAccess` from the second (the names are project-scoped, so they are prefixed with your ProjectName, e.g. `bank-marketing-prediction-`). If you get both, there is nothing to do — run the next cell.
>
> **Half 1 — subscribe, from the console.** Search the console for `quick`: you'll get two entries, **Amazon Quick** (the suite the BI product now lives in) and **QuickSight** (the classic entry) — either works. On the form, set a globally unique account name and a notification email, and leave **Authentication method** on its default (*Password-based or Single-Sign On*, the equivalent of `IAM_AND_QUICKSIGHT`) and **Encryption** on the AWS-managed key. **Default region** there is where QuickSight stores its own data, which is not the same thing as the identity region. The form has **no edition selector**, so re-run the first check afterwards rather than assuming — `STANDARD` has no Definition API and upgrades from *Manage QuickSight → Account settings*.
>
> **Half 2 — grant the service role access.** With the role now in existence, flip the stack parameter. Only the IAM nested stack changes, so this takes minutes rather than a full redeploy:
>
> ```bash
> STACK_NAME="${PROJECT_NAME:-bank-marketing-prediction}-workshop"
>
> OVERRIDES=""
> for k in $(aws cloudformation describe-stacks --stack-name "${STACK_NAME}" \
>              --query 'Stacks[0].Parameters[].ParameterKey' --output text); do
>   if [ "${k}" = "AttachQuickSightServiceRolePolicy" ]; then
>     OVERRIDES="${OVERRIDES} ParameterKey=${k},ParameterValue=true"
>   else
>     OVERRIDES="${OVERRIDES} ParameterKey=${k},UsePreviousValue=true"
>   fi
> done
>
> aws cloudformation update-stack --stack-name "${STACK_NAME}" \
>   --use-previous-template \
>   --capabilities CAPABILITY_NAMED_IAM CAPABILITY_AUTO_EXPAND \
>   --parameters ${OVERRIDES}
> ```
>
> The workshop docs (`model-governance-at-scale-with-sagemaker-ai`, Lab 5E → *Before You Start*) walk both halves with screenshots and are the canonical version.
>
> **Automating both halves instead.** For provisioning accounts in bulk rather than working through the labs, `SUBSCRIBE_QUICKSIGHT=true QUICKSIGHT_NOTIFICATION_EMAIL=you@example.com ./scripts/deploy-workshop.sh` subscribes, waits for provisioning, confirms the service role, and deploys with the parameter set. It defaults to `false` because the subscription is billable and permanent.
>
> Two things this notebook never needs, but you do if you want to open the dashboard in the QuickSight UI yourself: a QuickSight **Author/Admin** user for your identity (*Manage QuickSight → Manage users*), and `QUICKSIGHT_IDENTITY_REGION=<region>` in `.env` if your account's identity region is not `us-east-1`.
>
> Full prerequisites + troubleshooting: see the README's **QuickSight prerequisites (one-time per account)** section.

## 2. Verify QuickSight Subscription

In [ ]:
# Same check the dashboard build runs internally, surfaced here so the two
# things that can block this notebook are named before anything is created.
# Both are account / IAM setup, so this reports and continues instead of raising.
subscription = dash.report_subscription(ACCOUNT_ID)
QUICKSIGHT_READY = subscription['subscribed']

## 3. Verify Inference Data in Athena

In [ ]:
# Build every dataset, Athena view, visual, the analysis and the dashboard in
# one shot — the same create_dashboard() entry point `main.py dashboard create`
# uses. Setup failures are reported with their fix and the notebook continues.
result = dash.build_dashboard(region=AWS_DEFAULT_REGION)

## 9. Publish Dashboard via Definition API

In [ ]:
CONFIRM_DELETE = False  # set True to delete all governance QuickSight resources

if CONFIRM_DELETE:
    outcome = dash.delete_dashboard(region=AWS_DEFAULT_REGION)
else:
    print('Cleanup skipped — set CONFIRM_DELETE = True to run.')